In [ ]:
import json
import math
import re
from datetime import datetime

# =============================================================================
# CLINICAL SAFETY KERNEL (CSK) - V 16.0
# -----------------------------------------------------------------------------
# ARCHITECT: Eng. Mai Abu Al Saud
# ROLE: Clinical Reasoning Simulation + Decision Support Architecture
# GOVERNANCE: Zero Overconfidence & Hallucination Suppression
# =============================================================================

class ClinicalSafetyKernel:

    def __init__(self, user_input="", age=25):
        self.architect = "Eng. Mai Abu Al Saud"
        self.raw_text = user_input
        self.text = user_input.lower()
        self.age = age

        # Zero Overconfidence Protocol settings
        self.confidence_ceiling = 0.85
        self.min_signal_threshold = 1.0

        # تحذيرات ملف 20 حرفياً كما طلبتِ
        self.warnings = [
            "CRITICAL: THIS SYSTEM IS FOR PHYSICIAN USE ONLY.",
            "SAFETY NOTICE: This is a clinical simulation and decision-support tool.",
            "LIABILITY: It does NOT provide medical diagnosis or treatment decisions.",
            f"Zero Overconfidence Mode: ENABLED (Capped at {self.confidence_ceiling*100}%)",
            f"Architect: {self.architect}"
        ]

        self.rules = {
            "cardiac": {"chest pain": 1.4, "pressure": 1.1, "st-elevation": 2.5, "troponin": 2.5, "ecg": 1.5},
            "infection": {"fever": 1.2, "sepsis": 2.0, "wbc high": 1.8, "procalcitonin": 2.2},
            "neurology": {"weakness": 1.2, "stroke": 2.0, "facial droop": 2.0, "slurred speech": 1.8},
            "gastro": {"bleeding": 1.5, "hematemesis": 2.0, "melena": 1.8, "vomiting": 1.0}
        }
        self.negations = ["no", "denies", "without", "not", "negative", "absent", "none"]

    def display_governance_header(self):
        """عرض واجهة الهوية والتحذيرات المستخرجة من ملف 20"""
        print("\n" + "="*80)
        print(f"   {self.architect.upper()} | CLINICAL SAFETY KERNEL V 16.0")
        print("="*80)

        print("\n[PATIENT SAFETY NOTICE]")
        print("-" * 50)
        for warning in self.warnings:
            print(f">> {warning}")

        ts = datetime.now().strftime('%H:%M:%S')
        print(f">> LLM ADAPTER SYNC @ {ts}")
        print("-" * 80)

    def run_engine(self):
        # تفعيل المقدمة والتحذيرات أولاً
        self.display_governance_header()

        # طلب السيناريو
        initial_input = input("\nPlease Enter Initial Clinical Scenario: ")
        self.text = initial_input.lower()

        while True:
            scores = self.process_logic()
            total = sum(scores.values())

            if total <= self.min_signal_threshold:
                print("\n[!] ANTI-HALLUCINATION: Clinical signal integrity too low.")
                user_input = input("ACTION REQUIRED: Provide more symptoms or data: ")
                if user_input.lower() in ['exit', 'quit']: break
            else:
                probs = {k: min(v/total, self.confidence_ceiling) for k, v in scores.items()}
                ranking = sorted(probs.items(), key=lambda x: x[1], reverse=True)
                print(f"\n🧠 [REASONING]: Focusing on {ranking[0][0].upper()}")
                user_input = input("👨‍⚕️ Enter Data / 'report' / 'json' / 'audit': ")

            if user_input.lower() == "report":
                self.show_report(ranking)
                break
            elif user_input.lower() == "json":
                print("\n[JSON DATA]:", self.export_json())
                continue

            self.text += " " + user_input.lower()

    def process_logic(self):
        scores = {k: 0.1 for k in self.rules}
        for system, signals in self.rules.items():
            for sig, weight in signals.items():
                if sig in self.text:
                    scores[system] += weight
        return scores

    def show_report(self, ranking):
        print("\n" + "="*80)
        print(f"   FINAL CLINICAL INTERPRETATION | ARCHITECT: {self.architect}")
        print("="*80)
        top_dx, top_prob = ranking[0]
        print(f"\n🏆 HIGHEST POSSIBILITY DIAGNOSIS: {top_dx.upper()}")
        print(f"PROBABILITY: {round(top_prob*100, 1)}% (Capped by Governance)")
        print(f"GOVERNANCE: Verified under Zero-Overconfidence Protocol")
        print("="*80)

    def export_json(self):
        return json.dumps({"architect": self.architect, "timestamp": str(datetime.now())}, indent=2)

if __name__ == "__main__":
    engine = ClinicalSafetyKernel()
    engine.run_engine()
